In [1]:
!pip install -q sentence-transformers

In [4]:
import pandas as pd

df = pd.read_csv(
    "clean_recipe_dataset.csv"
)

print(df.shape)

df.head()

(15009, 16)


,title,description,ingredients,instructions,keywords,cuisine,author,image_url,likes,bookmarks,comment_count,popularity_score,cook_steps,url,ingredient_count,recipe_text
0,bắp luộc,món ăn quen thuộc nhưng lại giàu dinh dưỡ...,3-5 trái bắp | ít muối,bắp lột ít vỏ bắp bên ngoài rửa sạch đ...,"trái bắp,ít muối",Việt Nam,Phan Bao Van,https://img-global.cpcdn.com/recipes/31335d1dd...,4,1,1,12,3,https://cookpad.com/vn/cong-thuc/15648188,2,bắp luộc món ăn quen thuộc nhưng lại già...
1,trà sữa thái,cách làm món trà sữa thái ngon tuyệt của nh...,"20 gr trà thái | 2,2 lít nước | 1 gói tha...",làm thạch: 5gr bột thạch hòa với 500ml n...,"trà thái,lít nước,gói thạch con cá dẻo...",Việt Nam,Gem,https://img-global.cpcdn.com/recipes/95bf87bec...,0,24,0,72,4,https://cookpad.com/vn/cong-thuc/2318780,7,trà sữa thái cách làm món trà sữa thái n...
2,sữa điều bí đỏ,mình nấu theo tổ tiên mâch bảo nên mònh k cân ...,bí đỏ | hạt điều | sữa tươi,bí đỏ luộc chín hạt điều ngâm 2 tiếng rửa sạch...,"bí đỏ,hạt điều,sữa tươi",Việt Nam,Ching Ching,https://img-global.cpcdn.com/recipes/f5d694786...,6,2,0,18,2,https://cookpad.com/vn/cong-thuc/16223698,3,sữa điều bí đỏ mình nấu theo tổ tiên mâch bảo ...
3,cháo hến,"tối ăn cơm còn dư, chỉ cần thêm nước và ít muố...",200 g hến đã tách vỏ | 2 chén cơm nguội | 20 g...,"cơm nguội còn dư, thêm 3 chén nước và ít muối,...","hến đã tách vỏ,nguội,hành ngò,hành phi,gia vị",Việt Nam,Annie Vo,https://img-global.cpcdn.com/recipes/01f6d22c5...,25,3,0,59,3,https://cookpad.com/vn/cong-thuc/17195856,5,"cháo hến tối ăn cơm còn dư, chỉ cần thêm nước ..."
4,pizza khoai tây hải sản,một chiếc pizza được làm từ đế khoai tây vừa m...,1 củ khoai tây | 1 con tôm | 2 khoanh mực ống ...,"khoai tây rửa sạch hấp chín, nghiền nhuyễn mịn...","khoai tây,tôm,mực ống,cà chua,rau củ (ớt chuôn...",Việt Nam,Mẹ Em Nghé,https://img-global.cpcdn.com/recipes/91619421c...,11,11,1,56,7,https://cookpad.com/vn/cong-thuc/16090192,8,pizza khoai tây hải sản một chiếc pizza được l...


In [1]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # remove separators noise
    text = re.sub(r"\|", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
def build_recipe_text(row):

    title = clean_text(row.get("title", ""))
    description = clean_text(row.get("description", ""))
    ingredients = clean_text(row.get("ingredients", ""))
    keywords = clean_text(row.get("keywords", ""))

    # boost title importance (rất quan trọng cho ranking)
    recipe_text = f"""
    TITLE: {title}
    DESCRIPTION: {description}
    INGREDIENTS: {ingredients}
    KEYWORDS: {keywords}
    """

    return recipe_text.strip()

In [8]:
import re
df["recipe_text"] = df.apply(build_recipe_text, axis=1)

df[["title", "recipe_text"]].head()

,title,recipe_text
0,bắp luộc,TITLE: bắp luộc\n DESCRIPTION: món ăn qu...
1,trà sữa thái,TITLE: trà sữa thái\n DESCRIPTION: cách ...
2,sữa điều bí đỏ,TITLE: sữa điều bí đỏ\n DESCRIPTION: mình n...
3,cháo hến,TITLE: cháo hến\n DESCRIPTION: tối ăn cơm c...
4,pizza khoai tây hải sản,TITLE: pizza khoai tây hải sản\n DESCRIPTIO...


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "keepitreal/vietnamese-sbert"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
texts = df["recipe_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/470 [00:00<?, ?it/s]

Embedding shape: (15009, 768)


In [13]:
import numpy as np
np.save("recipe_embeddings.npy", embeddings)

df.to_csv("recipe_with_text.csv", index=False, encoding="utf-8-sig")

print("Saved embeddings + dataset")

Saved embeddings + dataset


In [14]:
df[["title", "url"]].to_csv(
    "recipe_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

In [15]:
#test
from sklearn.metrics.pairwise import cosine_similarity

query = embeddings[0].reshape(
    1,
    -1
)

scores = cosine_similarity(
    query,
    embeddings
)[0]

top_idx = scores.argsort()[-10:][::-1]

df.iloc[top_idx][
    [
        "title",
        "url"
    ]
]

,title,url
0,bắp luộc,https://cookpad.com/vn/cong-thuc/15648188
3738,đậu bắp luộc chắm chao,https://cookpad.com/vn/cong-thuc/15647440
4142,cháo mực,https://cookpad.com/vn/cong-thuc/15235272
11925,mực xào mướp hương,https://cookpad.com/vn/cong-thuc/15225447
12497,nước chấm bánh tráng mắm nêm,https://cookpad.com/vn/cong-thuc/16289454
277,vịt nấu mì quảng (mì quảng vịt),https://cookpad.com/vn/cong-thuc/16461072
3266,vịt cỏ ram gừng xả,https://cookpad.com/vn/cong-thuc/12894396
3070,bánh trung thu(bánh dẻo sắc màu),https://cookpad.com/vn/cong-thuc/15529561
5365,mực hấp hành gừng,https://cookpad.com/vn/cong-thuc/15149765
2280,"sườn chay,đậu hủ chiên,xào giá và cà rốt",https://cookpad.com/vn/cong-thuc/16068460
